# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davidsoooooon/ML-Intern---Briones/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
from google.colab import userdata
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

feature_source = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                THEN gsc_impressions ELSE 0
            END) AS impressions_early,

            AVG(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                THEN NULLIF(gsc_avg_position, 0)
            END) AS avg_position_early,

            STDDEV_SAMP(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                THEN NULLIF(gsc_avg_position, 0)
            END) AS position_volatility_early,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                     AND gsc_impressions > 0
                THEN report_date
            END) AS active_days_early,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-15'
                                     AND DATE '2026-03-28'
                THEN gsc_impressions ELSE 0
            END) AS impressions_late

        FROM {MARCH_DAILY}
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-28'
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        *,
        CASE
            WHEN impressions_late < 0.80 * impressions_early
            THEN 1 ELSE 0
        END AS is_declining_proxy
    FROM page_windows
    WHERE impressions_early >= 100
""").df()

print(f"Pages in feature frame: {len(feature_source):,}")
display(feature_source.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages in feature frame: 75,526


,client_hash_id,content_hash_id,impressions_early,avg_position_early,position_volatility_early,active_days_early,impressions_late,is_declining_proxy
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,211.0,4.245075,2.342553,14,435.0,0
1,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1394.0,6.104078,1.173984,13,1314.0,0
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,237.0,4.542255,1.455799,14,77.0,1
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,385.0,4.351876,1.405632,14,325.0,0
4,client_62f4a7e64f5e0096,content_225dc9235023be5f,272.0,10.398054,4.968515,14,200.0,1


In [9]:
audit = feature_source.copy()

def quartile_bucket(series):
    percentile = series.rank(method="average", pct=True)

    return pd.cut(
        percentile,
        bins=[0, 0.25, 0.50, 0.75, 1.00],
        labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
        include_lowest=True
    )

# Signal 1: volume
audit["volume_bucket"] = quartile_bucket(
    audit["impressions_early"]
)

volume_table = (
    audit.groupby("volume_bucket", observed=True)
    .agg(
        n=("is_declining_proxy", "size"),
        median_impressions=("impressions_early", "median"),
        declining_rate=("is_declining_proxy", "mean")
    )
    .reset_index()
)

volume_table["declining_rate_pct"] = (
    100 * volume_table["declining_rate"]
).round(2)

volume_table = volume_table[
    ["volume_bucket", "n", "median_impressions", "declining_rate_pct"]
]

print("SIGNAL 1 — EARLY IMPRESSION VOLUME")
display(volume_table)


# Signal 2: position volatility
audit["volatility_bucket"] = quartile_bucket(
    audit["position_volatility_early"]
)

volatility_table = (
    audit.groupby("volatility_bucket", observed=True)
    .agg(
        n=("is_declining_proxy", "size"),
        median_volatility=("position_volatility_early", "median"),
        declining_rate=("is_declining_proxy", "mean")
    )
    .reset_index()
)

volatility_table["declining_rate_pct"] = (
    100 * volatility_table["declining_rate"]
).round(2)

volatility_table = volatility_table[
    ["volatility_bucket", "n", "median_volatility", "declining_rate_pct"]
]

print("SIGNAL 2 — EARLY POSITION VOLATILITY")
display(volatility_table)

SIGNAL 1 — EARLY IMPRESSION VOLUME


,volume_bucket,n,median_impressions,declining_rate_pct
0,Q1 lowest,18852,153.0,32.44
1,Q2,18910,359.0,33.42
2,Q3,18879,872.0,32.30
3,Q4 highest,18885,2995.0,36.99


SIGNAL 2 — EARLY POSITION VOLATILITY


,volatility_bucket,n,median_volatility,declining_rate_pct
0,Q1 lowest,18879,0.887404,29.91
1,Q2,18880,2.133802,36.97
2,Q3,18880,4.563536,33.80
3,Q4 highest,18880,9.412281,34.48


### Signal verdicts

**Signal 1 — Impression volume: MIXED**

The lowest-volume group had a declining rate of 32.44%, while the highest-volume group reached 36.99%. However, the middle groups did not follow a clear pattern, moving from 33.42% to 32.30%. This tells me that impression volume may help, but it cannot reliably explain decline on its own.

**Signal 2 — Position volatility: MIXED**

The declining rate was 29.91% for the lowest-volatility group and 34.48% for the highest. Still, the rate jumped to 36.97% in Q2 and then dropped to 33.80% in Q3. Volatility may be useful, but the results are not consistent enough to treat it as a strong signal by itself.

### My baseline rule

My rule puts pages in the review queue when both their early impressions and position volatility are within the highest 25% of the dataset.

The score is calculated as:

`volume percentile × volatility percentile`

A higher score means that a page has strong search visibility but unstable rankings. It helps editors decide which pages to check first, but it does not automatically mean that the content needs to be changed.

- **Reason code:** `HIGH_VOLUME_UNSTABLE_POSITION`
- **Action label:** `REVIEW_FOR_REFRESH`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
volume_cutoff = audit["impressions_early"].quantile(0.75)

volatility_cutoff = audit[
    "position_volatility_early"
].quantile(0.75)

audit["volume_strength"] = (
    audit["impressions_early"].rank(pct=True)
)

audit["volatility_strength"] = (
    audit["position_volatility_early"].rank(pct=True)
)

audit["rule_hit"] = (
    (audit["impressions_early"] >= volume_cutoff)
    & (
        audit["position_volatility_early"]
        >= volatility_cutoff
    )
)

audit["baseline_action_score"] = np.where(
    audit["rule_hit"],
    audit["volume_strength"] * audit["volatility_strength"],
    0.0
)

audit["reason_code"] = np.where(
    audit["rule_hit"],
    "HIGH_VOLUME_UNSTABLE_POSITION",
    ""
)

audit["action_label"] = np.where(
    audit["rule_hit"],
    "REVIEW_FOR_REFRESH",
    "MONITOR"
)

queue = (
    audit.sort_values(
        [
            "baseline_action_score",
            "impressions_early",
            "position_volatility_early"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

queue.insert(
    0,
    "baseline_rank",
    np.arange(1, len(queue) + 1)
)

print(f"Volume threshold: {volume_cutoff:,.2f}")
print(f"Volatility threshold: {volatility_cutoff:.2f}")
print(f"Flagged pages: {queue['rule_hit'].sum():,}")

# Baseline evaluation
k = min(50, len(queue))
precision_at_50 = queue.head(k)["is_declining_proxy"].mean()
base_rate = queue["is_declining_proxy"].mean()

print(f"Base declining rate: {base_rate:.3f}")
print(f"Baseline Precision@{k}: {precision_at_50:.3f}")

# Do not export the future outcome or proxy label.
export_columns = [
    "baseline_rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_early",
    "avg_position_early",
    "position_volatility_early",
    "active_days_early",
    "volume_strength",
    "volatility_strength"
]

queue_export = queue[export_columns].copy()

output_path = Path(
    "work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
queue_export.to_csv(output_path, index=False)

print(f"Wrote {len(queue_export):,} rows to {output_path}")
display(queue_export.head(10))

Volume threshold: 1,488.00
Volatility threshold: 6.28
Flagged pages: 1,402
Base declining rate: 0.338
Baseline Precision@50: 0.360
Wrote 75,526 rows to work/outputs/baseline_action_score.csv


,baseline_rank,client_hash_id,content_hash_id,baseline_action_score,reason_code,action_label,impressions_early,avg_position_early,position_volatility_early,active_days_early,volume_strength,volatility_strength
0,1,client_20259bd6705d81d4,content_aa412de014b806a0,0.972934,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,13803.0,24.350193,19.089022,14,0.987355,0.985394
1,2,client_20259bd6705d81d4,content_01985e746d5b26d0,0.967206,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,14878.0,31.737655,17.207939,14,0.988984,0.977979
2,3,client_20259bd6705d81d4,content_35c904673f215f0b,0.951139,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,9033.0,27.204620,17.035416,14,0.973254,0.977277
3,4,client_20259bd6705d81d4,content_e579a10848b4569b,0.945961,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,13682.0,33.556232,14.356910,14,0.987163,0.958262
4,5,client_20259bd6705d81d4,content_66296153ae89bce6,0.944555,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,9306.0,31.572551,15.676968,14,0.974545,0.969226
5,6,client_73cda7b4e4f265ea,content_9bcfb1e373c01b7a,0.944271,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,8941.0,20.317985,15.897262,14,0.972831,0.970643
6,7,client_e5c2aa26a8598242,content_3acc45f5ce928b32,0.940288,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,9349.0,11.560719,15.058867,14,0.974697,0.964698
7,8,client_20259bd6705d81d4,content_1f851827b37ec059,0.929498,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,10187.0,24.349850,13.645859,14,0.977888,0.950516
8,9,client_20259bd6705d81d4,content_d0baa8614c538ef0,0.928058,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,6308.0,30.324009,16.535109,14,0.952308,0.974536
9,10,client_20259bd6705d81d4,content_4fcda6bb5f7049cf,0.924449,HIGH_VOLUME_UNSTABLE_POSITION,REVIEW_FOR_REFRESH,5632.0,19.486437,17.598092,14,0.943642,0.979661


## 3. Top-10 review

In [13]:
top10 = queue[queue["rule_hit"]].head(10).copy()

assert len(top10) == 10, "The rule produced fewer than 10 flagged pages."

wrong_if = [
    "the volatility comes from mixed queries or devices rather than weakening content.",
    "the early impressions came from a temporary campaign or news spike.",
    "seasonality changed search demand, so refreshing the page would not fix it.",
    "a technical indexing or canonical issue is causing the movement.",
    "the page was already updated after the March 15 decision moment.",
    "SERP feature changes created the movement without a content problem.",
    "the average position hides very different query-level positions.",
    "the 14-day feature window is too short and the movement is random noise.",
    "the page serves navigational intent and does not need an editorial refresh.",
    "the page competes with another site page and needs consolidation instead."
]

top10["why_it_is_here"] = top10.apply(
    lambda row:
        f"{row['impressions_early']:,.0f} early impressions and "
        f"{row['position_volatility_early']:.2f} position volatility; "
        f"both passed the 75th-percentile thresholds.",
    axis=1
)

top10["what_would_make_it_wrong"] = wrong_if

for _, row in top10.iterrows():
    print(
        f"Rank {int(row['baseline_rank'])}: "
        f"{row['action_label']} — "
        f"{row['why_it_is_here']} "
        f"Wrong if {row['what_would_make_it_wrong']}"
    )

Rank 1: REVIEW_FOR_REFRESH — 13,803 early impressions and 19.09 position volatility; both passed the 75th-percentile thresholds. Wrong if the volatility comes from mixed queries or devices rather than weakening content.
Rank 2: REVIEW_FOR_REFRESH — 14,878 early impressions and 17.21 position volatility; both passed the 75th-percentile thresholds. Wrong if the early impressions came from a temporary campaign or news spike.
Rank 3: REVIEW_FOR_REFRESH — 9,033 early impressions and 17.04 position volatility; both passed the 75th-percentile thresholds. Wrong if seasonality changed search demand, so refreshing the page would not fix it.
Rank 4: REVIEW_FOR_REFRESH — 13,682 early impressions and 14.36 position volatility; both passed the 75th-percentile thresholds. Wrong if a technical indexing or canonical issue is causing the movement.
Rank 5: REVIEW_FOR_REFRESH — 9,306 early impressions and 15.68 position volatility; both passed the 75th-percentile thresholds. Wrong if the page was already 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
print("Lowest-scoring rows among the reviewed top ten:")

display(
    top10.tail(3)[
        [
            "baseline_rank",
            "content_hash_id",
            "baseline_action_score",
            "impressions_early",
            "position_volatility_early",
            "active_days_early"
        ]
    ]
)

RULE_INPUTS = {
    "impressions_early",
    "position_volatility_early"
}

FORBIDDEN_INPUTS = {
    "impressions_late",
    "is_declining_proxy",
    "leak_decline_ratio",
    "trend_direction",
    "trend_pct"
}

assert RULE_INPUTS.isdisjoint(FORBIDDEN_INPUTS)
assert not FORBIDDEN_INPUTS.intersection(queue_export.columns)

print("Leakage check passed.")
print("The rule uses only March 1–14 information.")
print("Future outcomes are used only for auditing and Precision@50.")

Lowest-scoring rows among the reviewed top ten:


,baseline_rank,content_hash_id,baseline_action_score,impressions_early,position_volatility_early,active_days_early
7,8,content_1f851827b37ec059,0.929498,10187.0,13.645859,14
8,9,content_d0baa8614c538ef0,0.928058,6308.0,16.535109,14
9,10,content_4fcda6bb5f7049cf,0.924449,5632.0,17.598092,14


Leakage check passed.
The rule uses only March 1–14 information.
Future outcomes are used only for auditing and Precision@50.


### Weak-pick review

Ranks 8–10 had the lowest scores in my top ten, so I treated them as the weakest picks. They may still be worth reviewing, but the results do not confirm that the content itself has a problem. Their ranking changes could simply come from seasonality, different search queries, SERP changes, or normal data noise.

Because of this, the baseline should only help editors decide what to check first—not automatically tell them to rewrite the content.

## Self-check

Before submitting, I confirmed the following:

- [x] Every section is complete, including my explanations and the code supporting them.
- [x] The notebook runs from top to bottom without errors.
- [x] No client names, URLs, or private search queries are included.
- [x] My conclusions use careful language and present the results as decision support.
- [x] The notebook is committed under `work/notebooks/` in my repository.